In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

# 1. Load the official 2026 dataset
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)

print("Dataset shape:", df.shape)
print("\nMissing values before preparation:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# 2. Prepare missing values
for col in df.columns:
    if col == "converted":
        continue
    if df[col].dtype == "object" or isinstance(df[col].dtype, pd.CategoricalDtype):
        df[col] = df[col].fillna("NA")
    else:
        df[col] = df[col].fillna(0.0)

# 3. Split the data exactly as instructed
df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=1
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

df_full_train = df_full_train.reset_index(drop=True)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train["converted"].values
y_val = df_val["converted"].values

# Features and target
def get_features(data):
    return data.drop(columns=["converted"])

# Train a DictVectorizer and Logistic Regression model
def train_model(train_data, val_data, C=1.0):
    X_train = get_features(train_data)
    X_val = get_features(val_data)

    dv = DictVectorizer(sparse=True)
    train_dicts = X_train.to_dict(orient="records")
    val_dicts = X_val.to_dict(orient="records")

    X_train_encoded = dv.fit_transform(train_dicts)
    X_val_encoded = dv.transform(val_dicts)

    model = LogisticRegression(
        solver="liblinear", C=C, max_iter=1000
    )
    model.fit(X_train_encoded, train_data["converted"].values)

    probabilities = model.predict_proba(X_val_encoded)[:, 1]
    return dv, model, probabilities

# 4. Question 1: Numerical feature AUC
print("\nQUESTION 1: NUMERICAL FEATURE AUC")
numeric_features = [
    "lead_score",
    "number_of_courses_viewed",
    "interaction_count",
    "annual_income"
]

for feature in numeric_features:
    score = roc_auc_score(y_train, df_train[feature])
    if score < 0.5:
        score = roc_auc_score(y_train, -df_train[feature])
    print(f"{feature}: {score:.3f}")

# 5. Question 2: Validation AUC
dv, model, val_probs = train_model(df_train, df_val, C=1.0)
auc_val = roc_auc_score(y_val, val_probs)
print("\nQUESTION 2: Validation AUC =", round(auc_val, 3))

# 6. Question 3: Precision and recall intersection
thresholds = np.arange(0.0, 1.001, 0.01)
pr_results = []

for threshold in thresholds:
    predictions = (val_probs >= threshold).astype(int)
    precision = precision_score(
        y_val, predictions, zero_division=0
    )
    recall = recall_score(
        y_val, predictions, zero_division=0
    )

    # Ignore thresholds where both metrics are zero
    if precision == 0 and recall == 0:
        continue

    pr_results.append(
        (threshold, precision, recall, abs(precision - recall))
    )

best_intersection = min(pr_results, key=lambda row: row[3])
print("\nQUESTION 3: Precision/recall intersection")
print("Threshold:", round(best_intersection[0], 2))
print("Precision:", round(best_intersection[1], 3))
print("Recall:", round(best_intersection[2], 3))

# 7. Question 4: Threshold with maximum F1
f1_results = []

for threshold in thresholds:
    predictions = (val_probs >= threshold).astype(int)
    f1 = f1_score(y_val, predictions, zero_division=0)
    f1_results.append((threshold, f1))

best_f1 = max(f1_results, key=lambda row: row[1])
print("\nQUESTION 4: Maximum F1")
print("Threshold:", round(best_f1[0], 2))
print("F1 score:", round(best_f1[1], 3))

# 8. Question 5: Five-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=1)
cv_scores = []

for train_idx, val_idx in kf.split(df_full_train):
    fold_train = df_full_train.iloc[train_idx]
    fold_val = df_full_train.iloc[val_idx]

    _, _, fold_probs = train_model(fold_train, fold_val, C=1.0)
    fold_auc = roc_auc_score(fold_val["converted"].values, fold_probs)
    cv_scores.append(fold_auc)

print("\nQUESTION 5: Five-fold CV")
print("Fold AUCs:", [round(s, 3) for s in cv_scores])
print("Mean AUC:", round(np.mean(cv_scores), 3))
print("Standard deviation:", round(np.std(cv_scores), 3))

# 9. Question 6: Tune C using the same five folds
print("\nQUESTION 6: Hyperparameter tuning")
tuning_results = []

for C in [0.000001, 0.001, 1]:
    scores = []

    for train_idx, val_idx in kf.split(df_full_train):
        fold_train = df_full_train.iloc[train_idx]
        fold_val = df_full_train.iloc[val_idx]

        _, _, fold_probs = train_model(fold_train, fold_val, C=C)
        fold_auc = roc_auc_score(
            fold_val["converted"].values, fold_probs
        )
        scores.append(fold_auc)

    mean_auc = np.mean(scores)
    std_auc = np.std(scores)

    tuning_results.append((C, mean_auc, std_auc))
    print(
        f"C={C}: mean AUC={mean_auc:.3f}, "
        f"std={std_auc:.3f}"
    )

# Highest mean AUC wins; ties go to lower std, then smaller C
tuning_results.sort(key=lambda x: (-x[1], x[2], x[0]))
print("\nBest C:", tuning_results[0][0])


Dataset shape: (5000, 9)

Missing values before preparation:
lead_source          148
industry             240
employment_status    194
location             208
annual_income        369
lead_score            35
dtype: int64

QUESTION 1: NUMERICAL FEATURE AUC
lead_score: 0.788
number_of_courses_viewed: 0.723
interaction_count: 0.765
annual_income: 0.608

QUESTION 2: Validation AUC = 0.732

QUESTION 3: Precision/recall intersection
Threshold: 0.63
Precision: 0.733
Recall: 0.725

QUESTION 4: Maximum F1
Threshold: 0.41
F1 score: 0.758

QUESTION 5: Five-fold CV
Fold AUCs: [np.float64(0.731), np.float64(0.73), np.float64(0.736), np.float64(0.741), np.float64(0.721)]
Mean AUC: 0.732
Standard deviation: 0.007

QUESTION 6: Hyperparameter tuning
C=1e-06: mean AUC=0.617, std=0.019
C=0.001: mean AUC=0.749, std=0.010
C=1: mean AUC=0.732, std=0.007

Best C: 0.001
